In [1]:
import numpy as np
import random

In [2]:
### Generate cards from 9 to 14 (ace) for all colors/symbols (0, 1, 2, 3)
def getDeck():
    return [(number, color) for color in range(4) for number in range(9, 15)]
    
print(getDeck())

[(9, 0), (10, 0), (11, 0), (12, 0), (13, 0), (14, 0), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (9, 2), (10, 2), (11, 2), (12, 2), (13, 2), (14, 2), (9, 3), (10, 3), (11, 3), (12, 3), (13, 3), (14, 3)]


In [3]:
### Shuffle the cards randomly. Each player gets 9 cards
### (so one player cannot be certain which cards the other player has)

def getShuffled(deck):
    D = set(deck)
    A = set(random.sample(deck, 8))
    B = set(random.sample(list(D - A), 8))
    C = D - A - B
    if len(A.intersection(B)) > 0: print("Shuffle error 1")
    if len(A.intersection(B)) > 0: print("Shuffle error 2")
    if len(A.intersection(C)) > 0: print("Shuffle error 3") 
    DS = A | B | C
    if not DS == D: print("Shuffle error 4")  
    return list(A), list(B), list(C)

p1, p2, notUsed, = getShuffled(getDeck())
print(p1)
print(p2)


[(13, 1), (11, 0), (11, 3), (10, 1), (13, 0), (12, 3), (11, 2), (14, 2)]
[(11, 1), (12, 1), (9, 3), (14, 1), (9, 2), (14, 0), (10, 0), (10, 2)]


In [4]:
class Player():
    def __init__(self, name):
        
        self.name = name
        self.cards = None
    
    ### -------------------------------------------------------------
    
    ### TO BE IMPLEMENTED - player's strategy 
    ### input: declared card, i.e., the card which is supposed
    ### to be the top card of the pile: If None - you can put any card you want because
    ### (a) it is the first turn (pile is empty) or (b) some cards were drawn in the previous turn)
    ### output: - player's true decision, player's declaration (if not equal - (s)he cheats)
    
    def putCard(self, declared_card):
        ### DO NOT REMOVE TRUE CARD cards.remove!!!
        ### return an object (not id): self.cards[id], not id
        ### for instance: return self.cards[0], self.cards[0] 
        ### IMPORTANT: If you want to draw cards instead of put, return "draw"
        ### for instance: return "draw" 
        return self.cards[0], self.cards[0] 
    
    ### TO BE IMPLEMENTED - Decide whether to check or not opponent's move (return True or False)
    def checkCard(self, opponent_declaration):
        pass
    
    ### Notification sent at the end of a round
    ### One may implement this method, capture data, and use it to get extra info
    ### -- checked = TRUE -> someone checked. If FALSE, the remaining inputs do not play any role
    ### -- iChecked = TRUE -> I decided to check my opponent (so it was my turn); 
    ###               FALSE -> my opponent checked and it was his turn
    ### -- iDrewCards = TRUE -> I drew cards (so I checked but was wrong or my opponent checked and was right); 
    ###                 FALSE -> otherwise
    ### -- revealedCard - some card (X, Y). Only if I checked.
    ### -- noTakenCards - number of taken cards
    def getCheckFeedback(self, checked, iChecked, iDrewCards, revealedCard, noTakenCards, log=True):
        if log: print("Feedback = " + self.name + " : checked this turn = " + str(checked) +
              "; I checked = " + str(iChecked) + "; I drew cards = " + 
                      str(iDrewCards) + "; revealed card = " + 
                      str(revealedCard) + "; number of taken cards = " + str(noTakenCards))
    
    
    ### -------------------------------------------------------------
    
    ### Init player's hand
    def startGame(self, cards):
        self.cards = cards
    
    ### Add some cards to player's hand (if (s)he checked opponent's move, but (s)he was wrong)
    def takeCards(self, cards_to_take):
        self.cards = self.cards + cards_to_take

In [5]:
# Some examplary random player

class RandomPlayer(Player):
    
    ### player's random strategy
    def putCard(self, declared_card):
        
        ### check if must draw
        if len(self.cards) == 1 and declared_card is not None and self.cards[0][0] < declared_card[0]:
            return "draw"
        
        ### player randomly decides which card put on the table
        card = random.choice(self.cards)
        declaration = card
        
        ### player randomly decides whether to cheat or not
        cheat = np.random.choice([True, False])
       
        ### if (s)he decides to cheat, (s)he randomly declares the card.
        if cheat:
            declaration = random.choice(self.cards)             
            
        ### Yet, declared card should be no worse than a card on the top of the pile . 
        if declared_card is not None and declaration[0] < declared_card[0]:
            declaration = (min(declared_card[0]+1,14), declaration[1])

        ### return the decision (true card) and declaration (player's declaration)
        return card, declaration
    
    ### randomly decides whether to check or not
    def checkCard(self, opponent_declaration):
        return np.random.choice([True, False])
    

In [6]:
class Game():
    def __init__(self, players, log = True):
        self.players = players
        self.deck = getDeck()
        self.player_cards = getShuffled(self.deck)
        self.game_deck = self.player_cards[0] + self.player_cards[1]
        
        self.cheats = [0, 0]
        self.moves = [0, 0]
        self.checks = [0, 0]
        self.draw_decisions = [0, 0]
        
        for i, cards in zip([0, 1], self.player_cards):
            self.players[i].startGame(cards.copy())
            if log:
                print("Player (" + str(i + 1) + "): " + self.players[i].name + " received:")
                print(self.players[i].cards)
        
        ### Which card is on top
        self.true_card = None
        ### Which card was declared by active player
        self.declared_card = None
        
        ### Init pile: [-1] = top card
        self.pile = []
        
        ### Which player moves
        self.player_move = np.random.randint(2)
        
    def takeTurn(self, log = True):
        
        self.player_move = 1 - self.player_move
        
        if log: 
            print("")
            print("")
            print("==== CURRENT STATE ================================")
            print("==== " + self.players[self.player_move].name + " MOVES ====")
            print("Player (0): " + self.players[0].name + " hand:")
            print(self.players[0].cards)
            print("Player (1): " + self.players[1].name + " hand:")
            print(self.players[1].cards)
            print("Pile: ")
            print(self.pile)
            print("Declared top card:")
            print(self.declared_card)
            print("")
            
        activePlayer = self.players[self.player_move]
        opponent = self.players[1 - self.player_move]
        self.moves[self.player_move] += 1
        
        self.previous_declaration = self.declared_card
        decision = activePlayer.putCard(self.declared_card)
        
        if decision == "draw":
            
            if log: print("[+] " + activePlayer.name + " decides to draw cards")
            
            self.draw_decisions[self.player_move] += 1
            
            toTake = self.pile[max([-3, -len(self.pile)]):]
            for c in toTake: self.pile.remove(c)
            activePlayer.takeCards(toTake)
            for c in toTake: self.player_cards[self.player_move].append(c)
            
            self.declared_card = None
            self.true_card = None
            
            activePlayer.getCheckFeedback(False, False, False, None, None, log)
            opponent.getCheckFeedback(False, False, False, None, None, log)

        else:
            self.true_card, self.declared_card = decision
            if self.true_card != self.declared_card: self.cheats[self.player_move] += 1
            
            if log: print("[+] " + activePlayer.name + " puts " + str(self.true_card) +
                          " and declares " + str(self.declared_card))
        
            if not self.debugMove(): return False, self.player_move
        
            activePlayer.cards.remove(self.true_card)
            self.player_cards[self.player_move].remove(self.true_card) 
            self.pile.append(self.true_card)
        
            if opponent.checkCard(self.declared_card):
                
                self.checks[1 - self.player_move] += 1
                
                if log: print("[!] " + opponent.name + ": " + "I want to check")
                toTake = self.pile[max([-3, -len(self.pile)]):]
                for c in toTake: self.pile.remove(c)

                if not self.true_card == self.declared_card:
                    if log: print("\tYou are right!")
                    activePlayer.takeCards(toTake)
                
                    activePlayer.getCheckFeedback(True, False, True, None, len(toTake), log)
                    opponent.getCheckFeedback(True, True, False, tuple(toTake[-1]), len(toTake), log)
                
                    for c in toTake: self.player_cards[self.player_move].append(c)
                else:
                    if log: print("\tYou are wrong!")
                    opponent.takeCards(toTake)  
                
                    activePlayer.getCheckFeedback(True, False, False, None, len(toTake), log)
                    opponent.getCheckFeedback(True, True, True, tuple(toTake[-1]), len(toTake), log)
               
                    for c in toTake: self.player_cards[1 - self.player_move].append(c)
            
                if log:
                    print("Cards taken: ")
                    print(toTake)

                self.declared_card = None
                self.true_card = None
            else:
                activePlayer.getCheckFeedback(False, False, False, None, None, log)
                opponent.getCheckFeedback(False, False, False, None, None, log)

            
        if not self.debugGeneral(): return False, self.player_move
        return True, self.player_move
            
    def isFinished(self, log = True):
        if len(self.players[self.player_move].cards) == 0:
            if log: print(self.players[self.player_move].name + " wins!")
            return True
        return False

    def debugMove(self):
        if (self.previous_declaration is not None) and (self.true_card[0] < self.previous_declaration[0]) and \
                    len(self.players[self.player_move].cards) == 1:
            print("[ERROR] Last played card should be valid (it is revealed, you cannot cheat)!")
            return False
        if np.array(self.true_card).size != 2: 
            print("[ERROR] You put too many cards!")
            return False
        if self.true_card not in self.player_cards[self.player_move]:
            print("[ERROR] You do not have this card!")
            return False
        if self.true_card not in self.deck:
            print("[ERROR] There is no such card!")
            return False
        if (self.previous_declaration is not None) and len(self.pile) == 0:
            print("[ERROR] Inconsistency")
            return False
        if (self.previous_declaration is not None) and (self.declared_card[0] < self.previous_declaration[0]):
            print(len(self.pile))
            print(self.previous_declaration)
            print(self.declared_card)
            print(self.pile[-1])
            print("[ERROR] Improper move!")
            return False
        return True
    
    def debugGeneral(self):
        A = set(self.players[0].cards)
        B = set(self.players[1].cards)
        C = set(self.player_cards[0])
        D = set(self.player_cards[1])
        P = set(self.pile)
        E = set(self.game_deck)
        
        if not A == C: 
            print("Error 001")
            return False
        if not B == D:
            print("Error 002")
            return False
        if not A | B | P == E:
            print("Error 003")
            print(A)
            print(B)
            print(P)
            print(E)
            return False
        return True

Analyze few moves...

In [8]:
player1 = SimplePlayer("Player A")
player2 = YourPlayer("Player B")
game = Game([player1, player2])

for i in range(100):
    game.takeTurn()

NameError: name 'SimplePlayer' is not defined

In [8]:
### Some debug players
class DrawPlayer(Player):
    
    ### player's random strategy
    def putCard(self, declared_card):
        return "draw"
    
    ### randomly decides whether to check or not
    def checkCard(self, opponent_declaration):
        return np.random.choice([False, False])
    
class SimplePlayer(Player):
    
    ### player's simple strategy
    def putCard(self, declared_card):
        
        ### check if must draw
        if len(self.cards) == 1 and declared_card is not None and self.cards[0][0] < declared_card[0]:
            return "draw"
        
        card = min(self.cards, key=lambda x: x[0])
        declaration = (card[0], card[1])
        if declared_card is not None:
            min_val = declared_card[0]
            if card[0] < min_val: declaration = (min(min_val + 1, 14), declaration[1])
        return card, declaration
    
    def checkCard(self, opponent_declaration):
        if opponent_declaration in self.cards: return True
        return np.random.choice([True, False], p=[0.3, 0.7])
        

In [77]:
### Perform a full game 100 times
stats_wins = [0, 0]
stats_moves = [0, 0]
stats_cheats = [0, 0]
stats_errors = [0, 0]
stats_cards = [0, 0]
stats_checks = [0, 0]
stats_draw_decisions = [0, 0]
stats_pile_size = 0

repeats = 100
errors = 0

for t in range(10_000):
    player1 = YourPlayer("Player A")
    player2 = SimplePlayer("Player B")
    game = Game([player1, player2], log = False)
    
    error = False
    while True:
        valid, player = game.takeTurn(log = False)
        if not valid:
            error = True
            stats_errors[player] += 1
            errors += 1
            break
        if game.isFinished(log = False):
            stats_wins[player] += 1
            break
            
    stats_pile_size += len(game.pile)
    if not error:
        for j in range(2):
            stats_moves[j] += game.moves[j]
            stats_cheats[j] += game.cheats[j]
            stats_checks[j] += game.checks[j]
            stats_draw_decisions[j] += game.draw_decisions[j]
            stats_cards[j] += len(game.player_cards[j])

stats_pile_size /= (repeats - errors)          
for j in range(2):
    stats_moves[j] /= (repeats - errors)
    stats_cheats[j] /= (repeats - errors)
    stats_checks[j] /= (repeats - errors)
    stats_draw_decisions[j] /= (repeats - errors)
    stats_cards[j] /= (repeats - errors)

    
print("Wins:")
print(stats_wins)
print("Moves:")
print(stats_moves)
print("Cards:")
print(stats_cards)
print("Pile size:")
print(stats_pile_size)
print("Checks:")
print(stats_checks)
print("Draw decisions:")
print(stats_draw_decisions)
print("Cheats:")
print(stats_cheats)
print("Errors:")
print(stats_errors)
print("Total errors:")
print(errors)

2
(12, 0)
(-1, 0)
(12, 0)
[ERROR] Improper move!
2
(14, 2)
(-1, 0)
(9, 2)
[ERROR] Improper move!
[ERROR] You do not have this card!
[ERROR] Last played card should be valid (it is revealed, you cannot cheat)!
4
(11, 1)
(-1, 0)
(11, 1)
[ERROR] Improper move!
[ERROR] You do not have this card!
6
(13, 1)
(-1, 0)
(13, 1)
[ERROR] Improper move!
[ERROR] Last played card should be valid (it is revealed, you cannot cheat)!
[ERROR] You do not have this card!
[ERROR] You do not have this card!
[ERROR] You do not have this card!
[ERROR] You do not have this card!
7
(14, 1)
(-1, 0)
(10, 1)
[ERROR] Improper move!
[ERROR] You do not have this card!
[ERROR] You do not have this card!
10
(13, 0)
(-1, 0)
(13, 0)
[ERROR] Improper move!
[ERROR] You do not have this card!
[ERROR] Last played card should be valid (it is revealed, you cannot cheat)!
[ERROR] You do not have this card!
[ERROR] Last played card should be valid (it is revealed, you cannot cheat)!
[ERROR] Last played card should be valid (it is 

In [ ]:
### Implement player's strategy. You can compare it with random player 
### (or some strategy implemented by one of you colleagues)
### Time limit per decision 0.01s !!!

class YourPlayer(Player):
    """Gracz w oszusta, którego strategia opiera się o liczbę kart w ręce oraz o figurę karty przeciwnika. 
    Macierz wypłat zmienia się w trakcie zadania przez, co dostosowywana jest taktyka pod nieznaną macierz wypłat.
    Gracz operuje na prawdopodobieństwach odnoszących się do wykonywania odpowiednich akcji. Strategią dominującą
    jest zagranie najniższej karty w momencie gdy możemy ją zagrać. Jeśli nasza najmniejsza karta ma figurę niższą
    od figury na stole gracz ma możliwość oszukania. Oszukanie w tym momencie to deklaracja najbliższej karty z ręki
    i postawienie karty najmniejszej. Gracz ma też możliwość nie oszukania i wystawienia karty najbliższej tej na stole.
    Jeśli nie ma karty do oszukania to deklaruje losową wyższą lub równą kartę zadeklarowanej.
    Dodatkowo zaimplementowano strategię, która dla dużej liczby kart w ręce sprawia, że gracz wybiera wyższe karty, aby
    potem zmuszać przeciwnika do ciągnięcia kart, lub oszukiwania.
    """
    def __init__(self, name: str) -> None:
        self.name: str = name
        self.cards: list[tuple[int, int]] = [] # edycja konstruktora klasy w celu pozbycia się komunikatów w IDE.
        self.seen: list[tuple[int, int]] = [] # dodanie tablicy z widzianymi kartami
    def lowestCardDeclaired(self, declared_card: tuple[int, int]) -> tuple[int, int]:
        
        """Funkcja do wybierania karty o wartości najbliższej karcie zadeklarowanej.

        Args:
            declared_card (tuple[int, int]): Zadeklarowana karta.
        
        Returns:
            tuple[int, int]: Najbliższa karta z ręki do zadeklarowanej.
        """
        def sorter(x: tuple[int, int]) -> int | float:
            """Metoda pomocnicza do wybierania karty o wartości najbliższej karcie zadeklarowanej.

            Args:
                x (tuple[int, int]): Karta do porównania 

            Returns:
                int|float: Wartość figury większej lub równej karcie zadeklarowanej 
            """
            if x[0] >= declared_card[0]:
                return x[0]
            return np.inf
        return min([(-1,0)] + self.cards, key= lambda x: sorter(x))
    def putCard(self, declared_card: tuple[int,int]|None) -> tuple[tuple[int,int], tuple[int, int]]|str:
        """Określa decyzję gracza o wystawieniu karty poprawnej, oszukaniu lub ciągnięciu kart. Decyzja jest oparta na
        liczebności kart w ręce gracza. Strategia przyjmuje większe prawdopodobieństwo oszustwa dla małej liczby kart.
        Dodatkowo jeśli nasz gracz ma dużo kart to przyjmuje strategię wystawiania dużych wartości figur do zmuszania
        przeciwnika do ciągnięcia kart lub oszustwa. Dobieranie kart rozpatrujemy tylko wtedy kiedy zostaje nam jedna karta.
        Dobieranie jest strategią ściśle zdominowaną, ponieważ jak oszukamy to w najgorszym wypadku skończymy z jedną kartą,
        mniej niż przy dobieraniu.

        Args:
            declared_card (tuple[int,int] | None): Karta na stole.

        Returns:
            tuple[tuple[int,int], tuple[int, int]]|str: Deklaracja karty lub decyzja o ciągnięciu kart.
        """

        prob: float
        idx: int
        choiche: tuple[tuple[int,int], tuple[int, int]]
        minimal_card: tuple[int, int]
        fake_card: tuple[int, int]
        num_of_cards: int = len(self.cards)

        self.seen.extend(self.cards)
        self.seen = list(set(self.seen)) # kompresja listy widzianych kart

        # strategia zakładająca wystawianie wysokich kart w momencie, gdy mamy dużo kart
        if num_of_cards > 10:
            if declared_card is not None and len(np.array(self.cards)[np.array(self.cards)[:,0] >= declared_card[0]]) > 8:
                return sorted(self.cards, key=lambda x: x[0])[::-1][4], sorted(self.cards, key=lambda x: x[0])[::-1][4]
        
        minimal_card = min(self.cards, key=lambda x: x[0]) # znajdywanie najmniejszej karty

        # dostosowanie prawdopodobieństwa do liczby kart w ręce
        if declared_card is None or minimal_card[0] >= declared_card[0]:
            return minimal_card, minimal_card
        if num_of_cards < 2: # dla małej liczby kart taktyka bardziej ryzykowna
            prob = 0.8
        elif num_of_cards < 4: 
            prob = 0.6
        elif num_of_cards < 7:
            prob = 0.3
        elif num_of_cards < 10: # wraz ze wzrostem liczby kart zmniejszamy szansę na ruchy ryzykowne
            prob = 0.1
        else:
            prob = 0
        
        # wybór karty do potencjalnego oszukania
        fake_card = self.lowestCardDeclaired(declared_card)
        if fake_card[0] == -1: # jeśli nie jesteśmy w stanie wybrać karty do oszustwa, wybieramy losowo
            if num_of_cards != 1: # nie możemy oszukać ostatniej karty
               # dobieranie kart jest strategią ściśle zdominowaną
               random_card = np.random.randint(declared_card[0], 15), np.random.randint(0,4)
               i = 0
               while random_card in self.seen: # jeśli losowa karta była widziana powtarza losowanie
                   random_card = np.random.randint(declared_card[0], 15), np.random.randint(0,4)
                   if i == 1000: # wszystkie karty widziane, brak możliwości bezkarnego oszukania
                       break
                   i += 1
               return minimal_card, random_card
            return 'draw'
        
        idx = np.random.choice([0,1], p = [prob, 1-prob])
        choiche = [(minimal_card, fake_card), (fake_card, fake_card)][idx] # decyja czy wystawiamy kartę, czy oszukujemy
        return choiche[0], choiche[1]
    
    def checkCard(self, opponent_declaration: tuple[int, int]) -> bool:
        """Metoda do sprawdzania, czy przeciwnik oszukał czy nie. Decyzja jest podejmowana na podstawie figury przeciwnika.
        Funkcja zakłada, że przeciwnik jest bardziej skłonny do oszustwa przy deklaracji wysokiej karty. Stawianie wartości
        niskiej jest mało prawdopodobne do oszukania. Dodatkowo jest zabezpieczenie przed deklaracją karty zawartej w ręce.

        Args:
            opponent_declaration (tuple[int, int]): _description_

        Returns:
            bool: Decyzja o sprawdzeniu czy przeciwnik oszukał (True - oszukał, False - nie oszukał).
        """
        
        prob: float = 0
        if opponent_declaration in self.cards: 
            return True # sprawdzenie czy zadeklarowana karta przeciwnika jest w twojej ręce
        match opponent_declaration[0]: # wybór prawdopodobieństwa na bazie figury przeciwnika
            case 9:
                prob = 0
            case 10:
                prob = 0
            case 11:
                prob = 0.2
            case 12:
                prob = 0.5
            case 13:
                prob = 0.8
            case 14:
                prob = 0.9
        
        return np.random.choice([True, False], p = [prob, 1-prob])

In [ ]:
 num_of_cards = len(self.cards)
        if num_of_cards < 5:
            prob = 0.25
        elif num_of_cards < 11:
            prob = 0.1
        else:
            prob = 0.5